In [ ]:
### MP 7: Few-shot classification of tetracycline resistance (NARMS E. coli)
# 0. CPU-only, deterministic setup (INV-14/15)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import json, time, random, numpy as np, pandas as pd
random.seed(0); np.random.seed(0)
import torch; torch.manual_seed(0); torch.use_deterministic_algorithms(True, warn_only=True)
import mp7_core as M, mp7_narms as N
print(json.dumps(M.environment_report(), indent=2))

In [ ]:
## 1. Load the raw NARMS file (unmodified) and inspect its columns
RAW_PATH = "data/CVM-NARMS-Cecal.xlsx"          # FDA/USDA-FSIS NARMS cecal file, kept unmodified
RAW_SHA = M.sha256_file(RAW_PATH)
raw = N.load_raw(RAW_PATH)                        # ~1 min to read the 74,701 x 201 sheet
print(raw.shape, "sha256", RAW_SHA)
pd.set_option("display.max_rows", 200)
display(N.inspect_columns(raw))
print("auto-detected R/S/I columns:", N.detect_rsi_columns(raw))

In [ ]:
# Defaults are set for CVM-NARMS-Cecal.xlsx (see NarmsConfig docstring for every choice)
cfg = N.NarmsConfig()
print("STATE missing share:", raw["STATE"].isna().mean(), "-> no region feature")
print("TET MIC by SIR:", raw[raw.GENUS_NAME == "Escherichia"].groupby("TET SIR")["TET"].unique().to_dict())
df, drugs, clean_rep = N.prepare(raw, cfg)
print(json.dumps(clean_rep, indent=1, default=str))

In [ ]:
## 2. Temporal split, text records and leakage checks
TRAIN_MAX, TEST_MIN, N_EVAL = 2019, 2020, 400     # fix these in protocol.md before the sweep
train_df, test_df = N.temporal_split(df, cfg, TRAIN_MAX, TEST_MIN)
pool = N.to_examples(train_df, cfg, drugs)
test_all = N.to_examples(test_df, cfg, drugs)
ev = N.stratified_subsample(test_all, N_EVAL, seed=20261103)
N.assert_no_target_leak(pool + ev)       # no tetracycline-class drug anywhere in the records
N.assert_disjoint(pool, ev)              # no isolate on both sides
from collections import Counter
print("pool:", len(pool), Counter(e.label for e in pool))
print("test (all):", len(test_all), Counter(e.label for e in test_all))
print("test (evaluated sample):", len(ev), Counter(e.label for e in ev))
print("example record:\n", ev[0].text, "->", ev[0].label)
with open("test_ids.json", "w") as fh: json.dump([e.item_id for e in ev], fh)
import hashlib; EVAL_SHA = hashlib.sha256("\n".join(e.item_id for e in ev).encode()).hexdigest()

Leakage note: the cosine near-duplicate filter is **off** for tabular data. Different isolates often have identical
resistance profiles, and removing them would bias the pool. Leakage is controlled by the temporal split, disjoint
isolate ids, and excluding every tetracycline-class column.

In [ ]:
emb = M.Embedder("sentence-transformers/all-MiniLM-L6-v2", revision=None)   # TODO: pin revision sha
quota = M.QuotaLog("quota_log.jsonl", request_cap=None)
llm = M.HFBackend("Qwen/Qwen2.5-0.5B-Instruct", revision=None, quota_log=quota)
print("embedder:", emb.model_id, "| LLM:", llm.model_id, "@", llm.revision)
llm.check_verbalizers([M.VERBALIZERS[l] for l in M.LABELS])

In [ ]:
## 3. Contract check, prompt preview and timing
import deal
class Broken(M.Classifier):
    def _fit(self, demos): pass
    def _proba(self, texts): return np.full((len(texts), 3), 1/3)   # wrong shape -> must fail
try:
    Broken().fit([]).predict_proba(["x"]); print("NOT ENFORCED")
except deal.ContractError as e:
    print("contract enforced ->", e)

demo8 = M.sample_demos(pool, 8, seed=0)
c = M.FewShotLLMClassifier(llm, seed=0).fit(demo8)
print(c.render_example(ev[0].text)[-2000:])
t = time.time(); print(c.predict_proba([e.text for e in ev[:5]]).round(3))
print(f"{(time.time()-t)/5:.2f}s per item -> full sweep ~ {(time.time()-t)/5*len(ev)*25/60:.0f} min for the LLM")

In [ ]:
## 4. Shot sweep (writes predictions.jsonl)
KS, SEEDS = (0, 1, 3, 5, 8), (0, 1, 2, 3, 4)
factories = {
    "prototype":   lambda s: M.PrototypeClassifier(emb, temperature=0.05),
    "knn":         lambda s: M.KNNClassifier(emb, n_neighbors=5, temperature=0.05),
    "fewshot_llm": lambda s: M.FewShotLLMClassifier(llm, seed=s),
}
t0 = time.time()
runs = M.run_sweep(factories, pool, ev, KS, SEEDS, out_path="predictions.jsonl",
                   eval_sha256=EVAL_SHA, grading_eligible=False)   # INV-12 graded path N/A (instructor clarification)
print(f"sweep wall time {(time.time()-t0)/60:.1f} min")
quota.flush(note="MP7 sweep", model=llm.model_id, revision=llm.revision, hosted_api_requests=0)
json.dump(runs, open("runs_manifest.json", "w"), indent=1)

In [ ]:
## 5. Results audit: every number re-derived from predictions.jsonl
table, best, per_class = M.summarize("predictions.jsonl", metric=M.dev_macro_f1, n_boot=1000)
table["cell"] = table.apply(lambda r: f"{r.macro_f1_mean:.3f} ± {r.macro_f1_std:.3f} [{r.ci95_lo:.3f}, {r.ci95_hi:.3f}]", axis=1)
display(table.pivot(index="classifier", columns="k", values="cell"))
print("best:", best.classifier, "k =", best.k)
display(per_class.round(3))      # resistant-class recall is the proposal's priority metric
table.to_csv("results_table.csv", index=False); per_class.to_csv("per_class_best.csv", index=False)

In [ ]:
# Majority-label baseline on the same test sample (reference line for the report)
maj = Counter(e.label for e in pool).most_common(1)[0][0]
print("majority-class macro-F1:", round(M.dev_macro_f1([e.label for e in ev], [maj]*len(ev)), 3), "(predicts", maj, ")")
# Failure-mode diagnostics per run
diag = pd.DataFrame(runs)[["classifier","k","seed","max_pred_share","kl_pred_vs_gold",
                           "share_pred_eq_last_demo","gold_share_last_demo_label"]]
display(diag.groupby(["classifier","k"]).mean(numeric_only=True).round(3))

In [ ]:
rep = M.environment_report(); rep.update({"raw_sha256": RAW_SHA, "eval_ids_sha256": EVAL_SHA,
       "llm": [llm.model_id, llm.revision], "embedder": [emb.model_id, emb.revision],
       "split": {"train_max_year": TRAIN_MAX, "test_min_year": TEST_MIN, "n_eval": len(ev)},
       "cleaning": clean_rep, "quota": {"requests": quota.requests, "prompt_tokens": quota.prompt_tokens}})
json.dump(rep, open("environment.json", "w"), indent=2, default=str); print(json.dumps(rep, indent=2, default=str))